## 0. Imports

In [5]:
import json
import re
from datetime import date

import os
from pathlib import Path
from dotenv import load_dotenv


import numpy as np
import pandas as pd

## 1. Load Raw Data

In [6]:
load_dotenv()

root = Path(os.getenv("DATA_ROOT"))
mold_list       = root/ "WWW Bottom Mold list- All brands - 2026 (2).xlsx"
management_file = root/ "Mold Capacity Management.xlsm"

mold_list_raw_df             = pd.read_excel(mold_list,       sheet_name="All Brands -done")
vendor_master_raw_df         = pd.read_excel(mold_list,       sheet_name="Vendor Master")
mold_style_raw_df            = pd.read_excel(management_file, sheet_name="Style Mold Info")
mold_vendor_allcation_raw_df = pd.read_excel(management_file, sheet_name="Mold Vendor Allocation")
special_mold_raw_df          = pd.read_excel(management_file, sheet_name="Special Molds")

c:\Users\leta\VScode-Workspace\osms-mold-master-data\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


## 2. Helper Functions

In [7]:
# --- DataFrame cleaning ---

def clean_df_col_names(df):
    cols = {}
    for col in df.columns:
        new_col = str(col).split("\n")[0].strip()
        cols[col] = new_col.strip().lower().replace(' ', '_').replace('-', '_')
    df.rename(columns=cols, inplace=True)
    return df


def replace_whitespace(df):
    return df.replace(
        to_replace=[
            r"^\s*$",   # empty or whitespace-only
            r"^NA$",     # NA
            r"^N/A$",    # N/A
            r"^\xa0$",  # non-breaking space
        ],
        value=np.nan,
        regex=True,
    )


def normalize_col_name(col_name):
    return str(col_name).split("\n")[0].strip().lower().replace(" ", "_").replace("-", "_")


def pick_col(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


# --- Value normalisers ---

def none_if_nan(x):
    if pd.isna(x):
        return None
    return x


def to_int_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return int(float(x))
    except Exception:
        return x


def to_float_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x


def normalize_numeric_or_text(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        f = float(x)
        return int(f) if f.is_integer() else f
    except Exception:
        return str(x).strip()


def normalize_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    s = str(s).strip().upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or None


def normalize_mold_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    return str(s).strip().upper()


def normalize_season(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().upper()
    replacements = {"SPR": "S", "SS": "S", "AW": "F", "FW": "F"}
    for old, new in replacements.items():
        if value.startswith(old):
            value = value.replace(old, new, 1)
    return value if re.match(r'^[SF]\d{2}$', value) else np.nan


def normalize_location(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().title()
    aliases = {
        "North Vietnam": "Vietnam",
        "South Vietnam": "Vietnam",
        "Viet Nam": "Vietnam",
        "Indonesia ": "Indonesia",
    }
    return aliases.get(value, value)


def normalize_location_code(location_name):
    location_name = none_if_nan(location_name)
    if location_name is None:
        return None
    key = str(location_name).strip().title()
    location_code_map = {
        "Vietnam": "VN", "Indonesia": "ID", "India": "IN", "China": "CN",
        "Cambodia": "KH", "Thailand": "TH", "Myanmar": "MM", "Bangladesh": "BD",
        "Pakistan": "PK", "Philippines": "PH", "Taiwan": "TW", "Korea": "KR",
        "South Korea": "KR", "Japan": "JP", "Usa": "US", "United States": "US",
        "Mexico": "MX",
    }
    if key in location_code_map:
        return location_code_map[key]
    fallback = re.sub(r"[^A-Za-z]", "", key).upper()
    return fallback[:2] if fallback else None

LOCATION_CODE_TO_NAME = {
    "VN": "Vietnam",    "CN": "China",         "ID": "Indonesia",
    "BD": "Bangladesh", "KH": "Cambodia",      "TH": "Thailand",
    "MM": "Myanmar",    "IN": "India",          "PK": "Pakistan",
    "PH": "Philippines","TW": "Taiwan",         "KR": "South Korea",
    "JP": "Japan",      "US": "United States",  "MX": "Mexico",
}


def component_code_from_name(component_name):
    s = none_if_nan(component_name)
    if s is None:
        return None
    s = str(s).strip()
    m = re.match(r"^([^\s(]+)", s)
    return m.group(1) if m else s


def ensure_list(x):
    x = none_if_nan(x)
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [str(x)]


# Size columns: "1", "1.5", ..., "18"
size_cols = [str(x).replace(".0", "") for x in np.arange(1, 18.5, 0.5)]

## 3. Clean & Transform DataFrames

### 3a. Mold List -> `df`

In [8]:
df = mold_list_raw_df.copy()
df = clean_df_col_names(df)
df = replace_whitespace(df)

# Cast columns to target dtypes
dtype_set = {
    'brand': str, 'part_name': str, 'mold_code': str, 'compound': str,
    'vendor_name': str, 'location': str, 'mold_shop': str, 'season': str,
    'a_mold_cost': float, 'last_demand_season': str, 'mold_ownership': str,
    'mold_condition': str, 'daily_output': float, 'total_mold_qty': float,
    '1': float, '1.5': float, '2': float, '2.5': float, '3': float, '3.5': float,
    '4': float, '4.5': float, '5': float, '5.5': float, '6': float, '6.5': float,
    '7': float, '7.5': float, '8': float, '8.5': float, '9': float, '9.5': float,
    '10': float, '10.5': float, '11': float, '11.5': float, '12': float,
    '17.5': float, '18': float,
    'comments': str, 'remark': str, 'actual_output': str,
}
for col, dtype in dtype_set.items():
    if col in df.columns and dtype == str:
        df[col] = df[col].astype(dtype)
    elif col in df.columns and dtype == float:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")

# Normalize seasons, brand list, vendor name, location
df['season']             = df['season'].apply(normalize_season)
df['last_demand_season'] = df['last_demand_season'].apply(normalize_season)
df['brand']              = df['brand'].apply(lambda x: x.split("/"))
df['vendor_name']        = df['vendor_name'].str.strip().str.upper()
df['location']           = df['location'].apply(normalize_location)

# Keep only Saucony Outsole / Midsole rows
df = df[
    (df['brand'].apply(lambda x: 'Saucony' in x))
    & (df['part_name'].isin(['Outsole', 'Midsole']))
]

### 3b. Style-Mold mapping -> `mold_style_df`

In [9]:
mold_style_df = mold_style_raw_df.iloc[:, :4].copy()
mold_style_df.columns = ['Brand', 'Style_Name', 'Outsole_Name', 'Midsole_Name']
mold_style_df = replace_whitespace(mold_style_df)
mold_style_df = mold_style_df.dropna(subset=['Brand', 'Style_Name'], how='all')
# All rows are kept — the mismatch case (OS family ≠ MS family) is now handled
# correctly in the style map builder (section 4b) via soleTypes.

### 3c. Vendor allocation -> `mold_vendor_allocation_df`

In [10]:
mold_vendor_allocation_df = mold_vendor_allcation_raw_df.copy()
mold_vendor_allocation_df.columns = mold_vendor_allocation_df.iloc[0]
mold_vendor_allocation_df = mold_vendor_allocation_df[1:]
# Sole Type column dropped — component is identified by Sole Part alone in v3
mold_vendor_allocation_df = mold_vendor_allocation_df[
    ['Factory Number', 'Mold Code', 'Sole Part', 'Vendor ID']
]

## 4. Build Lookup Maps

### 4a. `vendor_reference_map`  (keyed by Vendor ID)

In [11]:
vendor_master_df = vendor_master_raw_df.copy()
vendor_master_df.columns = [normalize_col_name(c) for c in vendor_master_df.columns]
vendor_master_df = replace_whitespace(vendor_master_df)

vm_cols = set(vendor_master_df.columns)
vendor_id_col    = pick_col(vm_cols, ['vendor_id', 'vendor_no', 'factory_id', 'supplier_id', 'id'])
vendor_short_col = pick_col(vm_cols, ['vendor_short_name', 'vendor_shortname', 'short_name', 'vendor_code', 'vendor'])
vendor_full_col  = pick_col(vm_cols, ['vendor_name', 'vendor_full_name', 'supplier_name', 'factory_name', 'name'])
vendor_loc_col   = pick_col(vm_cols, ['location', 'country', 'vendor_location', 'factory_location', 'factory_country'])

# Keyed by Vendor ID (FTY number) — the stable, truly unique identifier.
# vendorCode (generated string) is not stored in v3.
vendor_reference_map = {}
if vendor_id_col is not None:
    for _, vm_row in vendor_master_df.iterrows():
        vendor_id = none_if_nan(vm_row.get(vendor_id_col))
        if vendor_id is None:
            continue
        vendor_id    = str(vendor_id).strip()
        short_name   = none_if_nan(vm_row.get(vendor_short_col)) if vendor_short_col else None
        full_name    = none_if_nan(vm_row.get(vendor_full_col))  if vendor_full_col  else None
        loc_raw      = none_if_nan(vm_row.get(vendor_loc_col))   if vendor_loc_col   else None
        loc_name     = normalize_location(loc_raw) if loc_raw else None
        loc_code     = normalize_location_code(loc_name) if loc_name else None
        if loc_code:
            loc_name = LOCATION_CODE_TO_NAME.get(loc_code, loc_name)

        vendor_reference_map[vendor_id] = {
            'name':      full_name or short_name,
            'shortName': short_name,
            'location':  {'code': loc_code, 'name': loc_name} if loc_code else None,
        }
else:
    print("Warning: vendor_id column not found in Vendor Master — reference.vendors will be empty.")

# Reverse lookup: upper-cased short name → vendor ID.
# Used to resolve vendorId from the vendor_name column in the mold list.
vendor_name_upper_to_id = {
    info['shortName'].strip().upper(): vid
    for vid, info in vendor_reference_map.items()
    if info.get('shortName')
}

# Fill missing vendor locations from mold list rows.
# Vendor master may not carry a location column; mold rows always do.
# Business rule: one vendor = one location, so first occurrence is definitive.
for _, row in df.iterrows():
    v_name = none_if_nan(row.get('vendor_name'))
    if v_name is None:
        continue
    vid = vendor_name_upper_to_id.get(v_name.strip().upper())
    if vid is None or vendor_reference_map[vid].get('location') is not None:
        continue
    loc_name = none_if_nan(row.get('location'))  # already normalized by step 3a
    loc_code = normalize_location_code(loc_name) if loc_name else None
    if loc_code:
        loc_name = LOCATION_CODE_TO_NAME.get(loc_code, loc_name)
        vendor_reference_map[vid]['location'] = {'code': loc_code, 'name': loc_name}

print(f"vendor_reference_map: {len(vendor_reference_map)} vendors")

vendor_reference_map: 74 vendors


### 4b. `style_map_by_mold`  (brand-keyed, with soleTypes)

In [12]:
# Intermediate: { moldCode → { brand → { styleName → set(soleTypes) } } }
# Sets accumulate 'Outsole' and/or 'Midsole' per (family, brand, style) tuple.
# Handles the mismatch case: a style using different families for OS vs MS now
# appears in each family's list with the correct soleTypes subset.
_raw_style_map = {}

for _, style_row in mold_style_df.iterrows():
    brand      = none_if_nan(style_row.get('Brand'))
    style_name = none_if_nan(style_row.get('Style_Name'))
    os_code    = normalize_mold_code(style_row.get('Outsole_Name'))
    ms_code    = normalize_mold_code(style_row.get('Midsole_Name'))

    if brand is None or style_name is None:
        continue

    if os_code:
        _raw_style_map             .setdefault(os_code, {})             .setdefault(brand, {})             .setdefault(style_name, set())             .add('Outsole')

    if ms_code:
        _raw_style_map             .setdefault(ms_code, {})             .setdefault(brand, {})             .setdefault(style_name, set())             .add('Midsole')

# Convert sets to sorted lists for JSON serialisation
style_map_by_mold = {
    mold_code: {
        brand: [
            {'styleName': sn, 'soleTypes': sorted(st)}
            for sn, st in styles.items()
        ]
        for brand, styles in brands.items()
    }
    for mold_code, brands in _raw_style_map.items()
}

print(f"style_map_by_mold: {len(style_map_by_mold)} mold families")

style_map_by_mold: 103 mold families


### 4c. `allocation_map_by_mold`  (per-component sourcing rules)

In [13]:
# { moldCode → { componentCode → [{factoryNumber, vendorId}] } }
# Sole Type is dropped — it is redundant when keyed by component code.
allocation_map_by_mold = {}

for _, alloc_row in mold_vendor_allocation_df.iterrows():
    mold_code_key  = normalize_mold_code(alloc_row.get('Mold Code'))
    sole_part      = none_if_nan(alloc_row.get('Sole Part'))
    component_code = component_code_from_name(sole_part)
    if mold_code_key is None or component_code is None:
        continue

    rule = {
        'factoryNumber': none_if_nan(alloc_row.get('Factory Number')),
        'vendorId':      none_if_nan(alloc_row.get('Vendor ID')),
    }
    comp_map  = allocation_map_by_mold.setdefault(mold_code_key, {})
    rule_list = comp_map.setdefault(str(component_code), [])
    if rule not in rule_list:
        rule_list.append(rule)

print(f"allocation_map_by_mold: {len(allocation_map_by_mold)} mold families")

allocation_map_by_mold: 29 mold families


### 4d. `special_sizing_by_mold_component`

Sheet columns = shoe sizes; cell values = mold sizes.

In [14]:
special_df = replace_whitespace(special_mold_raw_df.copy())
special_size_cols = [c for c in special_df.columns if str(c).strip().startswith('Size ')]

special_sizing_by_mold_component = {}
for _, sp_row in special_df.iterrows():
    mold_code_key      = normalize_mold_code(sp_row.get('Mold Code'))
    component_code_key = component_code_from_name(sp_row.get('Sole Part'))
    if mold_code_key is None or component_code_key is None:
        continue

    key = (mold_code_key, str(component_code_key))
    mold_to_shoes = special_sizing_by_mold_component.setdefault(key, {})

    for col in special_size_cols:
        shoe_size     = str(col).replace('Size', '').strip()  # col header = shoe size
        mold_size_raw = normalize_numeric_or_text(sp_row.get(col))  # cell value = mold size
        if mold_size_raw is None:
            continue
        try:
            mold_size_f = float(str(mold_size_raw))
        except (ValueError, TypeError):
            continue
        mold_size_key = str(int(mold_size_f)) if mold_size_f.is_integer() else str(mold_size_f)

        shoe_list = mold_to_shoes.setdefault(mold_size_key, [])
        if shoe_size not in shoe_list:
            shoe_list.append(shoe_size)

## 5. Build and Export JSON Result

In [15]:
result = {
    'schemaVersion': '3.0',
    'lastUpdated': date.today().isoformat(),
    'reference': {
        'vendors':          vendor_reference_map,   # keyed by vendorId (FTY)
        'moldShops':        {},
        'allowedSoleTypes': ['Outsole', 'Midsole'],
    },
    'families': {},
}

for _, row in df.iterrows():
    mold_code = none_if_nan(row.get('mold_code')) or none_if_nan(row.get('mold_code_note'))
    if mold_code is None:
        continue
    mold_code_key = normalize_mold_code(mold_code)

    sole_type      = none_if_nan(row.get('part_name')) or 'Unknown'
    component_name = none_if_nan(row.get('component_name'))
    compound       = none_if_nan(row.get('compound'))
    component_code = component_code_from_name(component_name)
    if component_code is None:
        continue

    vendor_name    = none_if_nan(row.get('vendor_name'))   # short name, upper-cased
    mold_shop_name = none_if_nan(row.get('mold_shop'))
    mold_shop_code = normalize_code(mold_shop_name) if mold_shop_name else None

    # Resolve vendorId from mold list vendor name via reverse lookup
    vendor_id = vendor_name_upper_to_id.get(vendor_name.strip().upper()) if vendor_name else None

    # Register mold shop in reference catalog
    if mold_shop_code and mold_shop_code not in result['reference']['moldShops']:
        result['reference']['moldShops'][mold_shop_code] = {'name': mold_shop_name}

    # ── Family block ──────────────────────────────────────────────────────────
    fam = result['families'].setdefault(str(mold_code), {
        'moldCode':               str(mold_code),
        'notes':                  None,
        'stylesUsingThisFamily':  {},
        'components':             {},
    })
    # Populate stylesUsingThisFamily once per family (first row wins)
    if not fam['stylesUsingThisFamily'] and mold_code_key in style_map_by_mold:
        fam['stylesUsingThisFamily'] = style_map_by_mold[mold_code_key]

    # ── Component block ───────────────────────────────────────────────────────
    sole_block = fam['components'].setdefault(str(sole_type), {})
    comp = sole_block.setdefault(str(component_code), {
        'displayName':   component_name,
        'compound':      compound,
        'notes':         None,
        'sizingRules':   {'moldSizeToShoeSizes': {}},
        'sourcingRules': [],
        'molds':         [],
    })
    if comp.get('compound') is None and compound is not None:
        comp['compound'] = compound

    # Populate sourcingRules once per component (first row wins)
    if not comp['sourcingRules']:
        alloc_for_family = allocation_map_by_mold.get(mold_code_key, {})
        comp['sourcingRules'] = alloc_for_family.get(str(component_code), [])

    # ── Sizing rules ──────────────────────────────────────────────────────────
    qty_by_mold_size = {
        s: to_int_or_none(row.get(s))
        for s in size_cols if pd.notna(row.get(s))
    }
    sizing_map = comp['sizingRules']['moldSizeToShoeSizes']
    for mold_size in qty_by_mold_size:
        sizing_map.setdefault(str(mold_size), None)

    sp_key           = (mold_code_key, str(component_code))
    sp_mold_to_shoes = special_sizing_by_mold_component.get(sp_key, {})
    if sp_mold_to_shoes:
        for mold_size, shoe_sizes in sp_mold_to_shoes.items():
            sizing_map[str(mold_size)] = shoe_sizes

    # ── Mold record ───────────────────────────────────────────────────────────
    comment = none_if_nan(row.get('comments'))
    remark  = none_if_nan(row.get('remark'))
    if comment and remark and comment != remark:
        combined_comment = f"{comment} | {remark}"
    else:
        combined_comment = comment or remark

    mold_entry = {
        'vendorId':         vendor_id,
        'moldShopCode':     mold_shop_code,
        'initSeason':       none_if_nan(row.get('season')),
        'lastDemandSeason': none_if_nan(row.get('last_demand_season')),
        'capacity': {
            'dailyOutput':  to_float_or_none(row.get('daily_output')),
            'actualOutput': to_float_or_none(row.get('actual_output')),
        },
        'asset': {
            'moldCost':      to_float_or_none(row.get('a_mold_cost')),
            'ownership':     none_if_nan(row.get('mold_ownership')),
            'condition':     none_if_nan(row.get('mold_condition')),
            'conditionNote': none_if_nan(row.get('mold_condition_note')),
        },
        'inventory': {
            'totalMoldQty':  to_int_or_none(row.get('total_mold_qty')),
            'qtyByMoldSize': qty_by_mold_size,
        },
        'comments': combined_comment,
    }
    # vendorName only present when vendorId is null (vendor pending FTY registration)
    if vendor_id is None and vendor_name:
        mold_entry['vendorName'] = vendor_name

    comp['molds'].append(mold_entry)

with open('../data/mold_data.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False, default=str)

print(f'Exported {len(result["families"])} mold families to mold_data.json  '
      f'(schemaVersion {result["schemaVersion"]})')

Exported 243 mold families to mold_data.json  (schemaVersion 3.0)
